In [132]:
!pip install optuna
from google.colab import drive
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from xgboost import XGBClassifier
import optuna
from sklearn.metrics import (accuracy_score, precision_score,
                            recall_score, f1_score, roc_auc_score)
import sklearn
sklearn.set_config(transform_output="pandas")

# Mount the drive to access files
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 13.5 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Load data
dir_path = '/content/drive/My Drive/Colab Notebooks/Spaceship Titanic/'
file_path = dir_path + 'train.csv'
train_data = pd.read_csv(file_path)

file_path = dir_path + 'test.csv'
test_data = pd.read_csv(file_path)

print(f"Train shape: {train_data.shape}, Test shape: {test_data.shape}")

Train shape: (8693, 14), Test shape: (4277, 13)


In [107]:
# Convert Transported into int
train_data['Transported'] = train_data['Transported'].astype(int)

train_data['dataset'] = 'train'
test_data['dataset'] = 'test'

data = pd.concat([train_data, test_data]).reset_index(drop=True)
data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,dataset
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,0.0,train
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,1.0,train
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,0.0,train
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,0.0,train
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,1.0,train


In [108]:
# Parse Cabin = deck/num/side
data['cabin_deck'] = data['Cabin'].str.split('/').str[0]
data['cabin_num'] = data['Cabin'].str.split('/').str[1]
data['cabin_side'] = data['Cabin'].str.split('/').str[2]

In [109]:
data.isna().sum()

,0
PassengerId,0
HomePlanet,288
CryoSleep,310
Cabin,299
Destination,274
Age,270
VIP,296
RoomService,263
FoodCourt,289
ShoppingMall,306


In [110]:
# Handle Missing Values

# Define numerical and categorical columns
numerical_col_list = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
categorical_col_list = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'cabin_side', 'Name']

# Fill numerical columns with median
for col in numerical_col_list:
    data[col] = data[col].fillna(data[col].median())

# Fill categorical columns with mode
for col in categorical_col_list:
    data[col] = data[col].fillna(data[col].mode()[0])

# Cabin Deck: fill with mode by HomePlanet & Destination
data['cabin_deck'] = data.groupby(['HomePlanet', 'Destination'])['cabin_deck'].transform(lambda x: x.fillna(x.mode()[0]))

<ipython-input-110-563302df63ce>:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[col] = data[col].fillna(data[col].mode()[0])


In [111]:
# Feature Engineering

# Passenger group count
data['passenger_group'] = data['PassengerId'].str.split('_').str[0]
data['passenger_group_count'] = data.groupby('passenger_group')['PassengerId'].transform('count')

# One-hot encoding: HomePlanet, Destination, cabin_deck, cabin_side
data = pd.get_dummies(data, columns=['HomePlanet', 'Destination', 'cabin_deck', 'cabin_side'],
                      prefix=['homeplanet', 'destination', 'deck', 'side'])

In [112]:
data.columns

Index(['PassengerId', 'CryoSleep', 'Cabin', 'Age', 'VIP', 'RoomService',
       'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name', 'Transported',
       'dataset', 'cabin_num', 'passenger_group', 'passenger_group_count',
       'homeplanet_Earth', 'homeplanet_Europa', 'homeplanet_Mars',
       'destination_55 Cancri e', 'destination_PSO J318.5-22',
       'destination_TRAPPIST-1e', 'deck_A', 'deck_B', 'deck_C', 'deck_D',
       'deck_E', 'deck_F', 'deck_G', 'deck_T', 'side_P', 'side_S'],
      dtype='object')

In [113]:
# Features
feature_list = ['CryoSleep', 'Age', 'VIP', 'RoomService', 'FoodCourt',
                'ShoppingMall', 'Spa', 'VRDeck', 'passenger_group_count'
                ] + \
                list(data.columns[data.columns.str.startswith('homeplanet')]) + \
                list(data.columns[data.columns.str.startswith('destination')]) + \
                list(data.columns[data.columns.str.startswith('deck')]) + \
                list(data.columns[data.columns.str.startswith('side')])
feature_list

['CryoSleep',
 'Age',
 'VIP',
 'RoomService',
 'FoodCourt',
 'ShoppingMall',
 'Spa',
 'VRDeck',
 'passenger_group_count',
 'homeplanet_Earth',
 'homeplanet_Europa',
 'homeplanet_Mars',
 'destination_55 Cancri e',
 'destination_PSO J318.5-22',
 'destination_TRAPPIST-1e',
 'deck_A',
 'deck_B',
 'deck_C',
 'deck_D',
 'deck_E',
 'deck_F',
 'deck_G',
 'deck_T',
 'side_P',
 'side_S']

In [114]:
# Split train & test data
new_train_data = data[data['dataset'] == 'train'].copy()
new_test_data = data[data['dataset'] == 'test'].reset_index(drop=True).copy()

In [136]:
# Prepare features and target variable
X = new_train_data[feature_list]
y = new_train_data['Transported']

# Prepare test data
X_test = new_test_data[feature_list]

X.shape

(8693, 25)

In [137]:
# Initialize XGBoost parameters
# Use optuna to find the best hyperparameters
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'n_estimators': 1400,
    'learning_rate': 0.04931476093184571,
    'max_depth': 4,
    'subsample': 0.6578843387359619,
    'colsample_bytree': 0.6902394255031622,
    'gamma': 0.11174367093570015,
    'random_state': 0,
    'early_stopping_rounds': 50
}

# Store results
metrics = {
    'accuracy': [],
    'precision': [],
    'recall': [],
    'f1': [],
    'roc_auc': []
}

# Training model
n_splits=5
kf = KFold(n_splits=n_splits)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
    print(f"\nFold {fold}/{n_splits}")

    # Split data
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Initialize and train model
    model = XGBClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=50,

    )

    # Predictions
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)
    y_proba = y_proba[:, 1]

    # Calculate metrics
    metrics['accuracy'].append(accuracy_score(y_val, y_pred))
    metrics['precision'].append(precision_score(y_val, y_pred, average='weighted'))
    metrics['recall'].append(recall_score(y_val, y_pred, average='weighted'))
    metrics['f1'].append(f1_score(y_val, y_pred, average='weighted'))

    # ROC AUC (handle multi-class)
    if len(np.unique(y)) > 2:
        metrics['roc_auc'].append(roc_auc_score(y_val, y_proba, multi_class='ovo'))
    else:
        metrics['roc_auc'].append(roc_auc_score(y_val, y_proba))

# Print results
print("\nFinal Metrics Across All Folds:")
for metric, values in metrics.items():
    print(f"{metric.capitalize()}: {np.mean(values):.4f} ± {np.std(values):.4f}")


Fold 1/5
[0]	validation_0-logloss:0.67815
[50]	validation_0-logloss:0.46153
[100]	validation_0-logloss:0.44087
[150]	validation_0-logloss:0.43656
[200]	validation_0-logloss:0.43482
[250]	validation_0-logloss:0.43355
[300]	validation_0-logloss:0.43231
[324]	validation_0-logloss:0.43284

Fold 2/5
[0]	validation_0-logloss:0.67912
[50]	validation_0-logloss:0.45450
[100]	validation_0-logloss:0.43311
[150]	validation_0-logloss:0.42827
[200]	validation_0-logloss:0.42466
[250]	validation_0-logloss:0.42369
[300]	validation_0-logloss:0.42168
[350]	validation_0-logloss:0.42179
[400]	validation_0-logloss:0.42230
[407]	validation_0-logloss:0.42164

Fold 3/5
[0]	validation_0-logloss:0.67908
[50]	validation_0-logloss:0.45815
[100]	validation_0-logloss:0.43410
[150]	validation_0-logloss:0.42417
[200]	validation_0-logloss:0.42265
[250]	validation_0-logloss:0.41963
[300]	validation_0-logloss:0.41948
[350]	validation_0-logloss:0.41939
[391]	validation_0-logloss:0.41977

Fold 4/5
[0]	validation_0-logloss

In [138]:
# Create feature importance DataFrame with proper names
feature_importances = pd.DataFrame({
    'feature': feature_list,
    'importance': model.feature_importances_  # Or mean_importances from cross-val
}).sort_values('importance', ascending=False)

# Display with clean formatting
print("\nFeature Importances:")
print(feature_importances.to_string(index=False))


Feature Importances:
                  feature  importance
                CryoSleep    0.418519
                   deck_G    0.059196
         homeplanet_Earth    0.058088
        homeplanet_Europa    0.050989
                      Spa    0.036990
                   deck_E    0.034678
              RoomService    0.033075
                   deck_F    0.032002
                   VRDeck    0.030620
                   side_S    0.024708
          homeplanet_Mars    0.023707
                FoodCourt    0.022825
                   side_P    0.022003
                   deck_B    0.020825
                   deck_C    0.019460
             ShoppingMall    0.019050
  destination_55 Cancri e    0.014825
                      Age    0.013722
destination_PSO J318.5-22    0.012148
                   deck_A    0.011528
  destination_TRAPPIST-1e    0.011496
                   deck_D    0.010768
    passenger_group_count    0.010257
                      VIP    0.008522
                   deck_T   

In [139]:
# Final model training
params = {
    'objective': 'binary:logistic',  # Change for multi-class
    'n_estimators': 1400,
    'learning_rate': 0.04931476093184571,
    'max_depth': 4,
    'subsample': 0.6578843387359619,
    'colsample_bytree': 0.6902394255031622,
    'gamma': 0.11174367093570015,
    'random_state': 0
}

final_model = XGBClassifier(**params)
final_model.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.6902394255031622, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, gamma=0.11174367093570015,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.04931476093184571,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1400, n_jobs=None,
              num_parallel_tree=None, random_state=0, ...)

In [140]:
# Prepare submission
test_preds = final_model.predict(X_test)

# Convert 0 or 1 into FALSE or TRUE
test_preds = test_preds.astype(bool)

submission_data = pd.DataFrame({
    'PassengerId': new_test_data['PassengerId'],
    'Transported': test_preds
})
submission_data.to_csv(dir_path+'final_submission_v2.csv', index=False)

In [134]:
# # Find the best hyperparameters

# ## Define the objective function for Optuna
# def objective(trial):
#     params = {
#         'objective': 'binary:logistic',
#         'eval_metric': 'logloss',
#         'n_estimators': trial.suggest_int('n_estimators', 100, 2000, step=100),
#         'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.2),
#         'max_depth': trial.suggest_int('max_depth', 3, 15),
#         'subsample': trial.suggest_uniform('subsample', 0.6, 1.0),
#         'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.6, 1.0),
#         'gamma': trial.suggest_uniform('gamma', 0, 1),
#         'random_state': 0,
#         'early_stopping_rounds': 50
#     }

#     # Store the results for each fold
#     metrics = {
#         'accuracy': [],
#         'precision': [],
#         'recall': [],
#         'f1': [],
#         'roc_auc': []
#     }

#     # Cross-validation
#     n_splits = 5
#     kf = KFold(n_splits=n_splits)

#     for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
#         print(f"\nFold {fold}/{n_splits}")

#         # Split data
#         X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#         y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

#         # Initialize and train the model
#         model = XGBClassifier(**params)
#         model.fit(
#             X_train, y_train,
#             eval_set=[(X_val, y_val)],
#             verbose=0,
#         )

#         # Predictions
#         y_pred = model.predict(X_val)
#         y_proba = model.predict_proba(X_val)
#         y_proba = y_proba[:, 1]

#         # Calculate metrics
#         metrics['accuracy'].append(accuracy_score(y_val, y_pred))
#         metrics['precision'].append(precision_score(y_val, y_pred, average='weighted'))
#         metrics['recall'].append(recall_score(y_val, y_pred, average='weighted'))
#         metrics['f1'].append(f1_score(y_val, y_pred, average='weighted'))

#         # ROC AUC (handle multi-class)
#         if len(np.unique(y)) > 2:
#             metrics['roc_auc'].append(roc_auc_score(y_val, y_proba, multi_class='ovo'))
#         else:
#             metrics['roc_auc'].append(roc_auc_score(y_val, y_proba))

#     # Return the metric you want to optimize; here, we use the F1-score
#     return np.mean(metrics['f1'])

# # Create a study to optimize the hyperparameters
# study = optuna.create_study(direction='maximize')  # 'maximize' for optimization metrics like accuracy, f1, etc.
# study.optimize(objective, n_trials=100)  # Run 100 trials

# # Output the best hyperparameters
# print("Best hyperparameters found: ", study.best_params)

# # You can also print the best value for the objective
# print("Best F1 score: ", study.best_value)

[I 2025-04-28 15:11:11,015] A new study created in memory with name: no-name-eea9a415-d907-4ff9-9953-94683cc4ea84
<ipython-input-134-1caae69baf53>:9: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.2),
<ipython-input-134-1caae69baf53>:11: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'subsample': trial.suggest_uniform('subsample', 0.6, 1.0),
<ipython-input-134-1caae69baf53>:12: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'colsample_bytree': trial.suggest_uniform('colsample_bytree',


Fold 1/5

Fold 2/5

Fold 3/5

Fold 4/5

Fold 5/5


[I 2025-04-28 15:11:16,372] Trial 0 finished with value: 0.8026503196664804 and parameters: {'n_estimators': 1100, 'learning_rate': 0.10752431956969061, 'max_depth': 4, 'subsample': 0.8231221435744875, 'colsample_bytree': 0.7244682734079122, 'gamma': 0.7476367454901625}. Best is trial 0 with value: 0.8026503196664804.
<ipython-input-134-1caae69baf53>:9: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.2),
<ipython-input-134-1caae69baf53>:11: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'subsample': trial.suggest_uniform('subsample', 0.6, 1.0),
<ipython-input-134-1caae69baf53>:12: FutureWarning: suggest_uniform has been depr


Fold 1/5

Fold 2/5

Fold 3/5

Fold 4/5

Fold 5/5


[I 2025-04-28 15:11:21,139] Trial 1 finished with value: 0.7953402745765932 and parameters: {'n_estimators': 600, 'learning_rate': 0.16044569225929964, 'max_depth': 9, 'subsample': 0.6554512452963498, 'colsample_bytree': 0.8282877031602639, 'gamma': 0.5394008008548054}. Best is trial 0 with value: 0.8026503196664804.
<ipython-input-134-1caae69baf53>:9: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.2),
<ipython-input-134-1caae69baf53>:11: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'subsample': trial.suggest_uniform('subsample', 0.6, 1.0),
<ipython-input-134-1caae69baf53>:12: FutureWarning: suggest_uniform has been depre

...


Fold 1/5

Fold 2/5

Fold 3/5

Fold 4/5

Fold 5/5


[I 2025-04-28 15:20:36,383] Trial 98 finished with value: 0.7991423803892884 and parameters: {'n_estimators': 1900, 'learning_rate': 0.04169178876377865, 'max_depth': 3, 'subsample': 0.7968932738937696, 'colsample_bytree': 0.698932128553263, 'gamma': 0.8792529513798195}. Best is trial 32 with value: 0.8041658262994972.
<ipython-input-134-1caae69baf53>:9: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.2),
<ipython-input-134-1caae69baf53>:11: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'subsample': trial.suggest_uniform('subsample', 0.6, 1.0),
<ipython-input-134-1caae69baf53>:12: FutureWarning: suggest_uniform has been dep


Fold 1/5

Fold 2/5

Fold 3/5

Fold 4/5

Fold 5/5


[I 2025-04-28 15:20:38,123] Trial 99 finished with value: 0.7968905902300708 and parameters: {'n_estimators': 100, 'learning_rate': 0.05639657060353712, 'max_depth': 4, 'subsample': 0.7485310395093112, 'colsample_bytree': 0.8404840004797264, 'gamma': 0.21168520861294043}. Best is trial 32 with value: 0.8041658262994972.


Best hyperparameters found:  {'n_estimators': 1400, 'learning_rate': 0.04931476093184571, 'max_depth': 4, 'subsample': 0.6578843387359619, 'colsample_bytree': 0.6902394255031622, 'gamma': 0.11174367093570015}
Best F1 score:  0.8041658262994972


In [135]:
study.best_params

{'n_estimators': 1400,
 'learning_rate': 0.04931476093184571,
 'max_depth': 4,
 'subsample': 0.6578843387359619,
 'colsample_bytree': 0.6902394255031622,
 'gamma': 0.11174367093570015}